## Setup
Required packages: numpy, os, tifffile, stardist, matplotlib. 
Stardist also has some strange dependecy requirements; highly recommend setting up new conda environment (conda create env -n stardist) before pip installing.

Goals: Use manually segmented masks to optimize parameters, then run on whole dataset.

In [ ]:
#if set up env called stardist, can activate here
#%conda init
#%conda activate stardist_env

In [ ]:
#install all dependencies - RUN THIS CELL ONCE
#if doesn't work, try installing in console; just remove %

#stardist has dependency on tensorflow, make sure to use python 3.11 
# and install tensorflow first

%pip install tensorflow
%pip install stardist

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [50]:
#imports 
from stardist.models import StarDist2D
import numpy as np
from tifffile import imwrite, imread
from pathlib import Path
from skimage.measure import label

import os
import numpy as np
import tifffile
from stardist.models import StarDist2D
from stardist.matching import matching_dataset
from csbdeep.utils import normalize

import os
import numpy as np
import matplotlib.pyplot as plt
from tifffile import imread
from skimage.morphology import white_tophat, disk
from csbdeep.utils import normalize
from stardist.models import StarDist2D
import re

In [73]:
# Define your directory
data_dir = Path('manual_masks')
output_dir = data_dir / 'stardist_masks'
output_dir.mkdir(exist_ok=True)

# Find all manual segmentation files
seg_files = list(data_dir.glob('*_seg.npy'))

for f in seg_files:
    # Load the manual dictionary
    data = np.load(f, allow_pickle=True).item()
    
    # Extract the masks (integer labels: 0=bg, 1, 2, 3...)
    masks = data['masks'].astype(np.uint16)
    
    # Define new filename (e.g., cell_01_seg.npy -> cell_01_mask.tif)
    new_filename = f.name.replace('_seg.npy', '_star.tif')
    save_path = output_dir / new_filename
    
    # Save as TIFF
    imwrite(str(save_path), masks)
    print(f"Converted: {f.name} -> {new_filename}")

print(f"\nDone! All masks are ready for StarDist and saved to {output_dir}")

Converted: Phase_Hn249_00001_seg.npy -> Phase_Hn249_00001_star.tif
Converted: Phase_Hn249_00002_seg.npy -> Phase_Hn249_00002_star.tif
Converted: Phase_Hn249_00010_seg.npy -> Phase_Hn249_00010_star.tif
Converted: Phase_Hn249_00011_seg.npy -> Phase_Hn249_00011_star.tif
Converted: Phase_Hn249_00012_seg.npy -> Phase_Hn249_00012_star.tif
Converted: Phase_Hn249_00016_seg.npy -> Phase_Hn249_00016_star.tif
Converted: Phase_Hn249_00025_seg.npy -> Phase_Hn249_00025_star.tif
Converted: Phase_Hn249_00030_seg.npy -> Phase_Hn249_00030_star.tif

Done! All masks are ready for StarDist and saved to manual_masks\stardist_masks


In [74]:
#import
raw_dir = "fluorescence_data"
mask_dir = "manual_masks/stardist_masks"
save_dir = "stardist_masks"

# List files in the directory to verify what is actually there
print(os.listdir(raw_dir))
print(os.listdir(mask_dir))

def get_id(fname):
    match = re.search(r'(\d{5})', fname)
    return match.group(1) if match else None

raw_files = sorted([f for f in os.listdir(raw_dir) if f.endswith('.tif')])
mask_map = {get_id(m): m for m in os.listdir(mask_dir) if get_id(m)}

['Image_Hn249_150ms_(00001).tif', 'Image_Hn249_150ms_(00002).tif', 'Image_Hn249_150ms_(00003).tif', 'Image_Hn249_150ms_(00004).tif', 'Image_Hn249_150ms_(00005).tif', 'Image_Hn249_150ms_(00006).tif', 'Image_Hn249_150ms_(00007).tif', 'Image_Hn249_150ms_(00008).tif', 'Image_Hn249_150ms_(00009).tif', 'Image_Hn249_150ms_(00010).tif', 'Image_Hn249_150ms_(00011).tif', 'Image_Hn249_150ms_(00012).tif', 'Image_Hn249_150ms_(00013).tif', 'Image_Hn249_150ms_(00014).tif', 'Image_Hn249_150ms_(00015).tif', 'Image_Hn249_150ms_(00016).tif', 'Image_Hn249_150ms_(00017).tif', 'Image_Hn249_150ms_(00018).tif', 'Image_Hn249_150ms_(00019).tif', 'Image_Hn249_150ms_(00020).tif', 'Image_Hn249_150ms_(00021).tif', 'Image_Hn249_150ms_(00022).tif', 'Image_Hn249_150ms_(00023).tif', 'Image_Hn249_150ms_(00024).tif', 'Image_Hn249_150ms_(00025).tif', 'Image_Hn249_150ms_(00026).tif', 'Image_Hn249_150ms_(00027).tif', 'Image_Hn249_150ms_(00028).tif', 'Image_Hn249_150ms_(00029).tif', 'Image_Hn249_150ms_(00030).tif', 'Image_Hn

In [75]:
# 1. Initialize the model
model = StarDist2D.from_pretrained('2D_versatile_fluo')

# 2. Get file lists
raw_files = sorted([f for f in os.listdir(raw_dir) if f.endswith('.tif')])
mask_files = sorted([f for f in os.listdir(mask_dir) if f.endswith('.tif')])


Found model '2D_versatile_fluo' for 'StarDist2D'.
Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.479071, nms_thresh=0.3.


In [98]:
# --- 1. SETUP AGGREGATE LISTS ---
all_gt_masks = []
all_pred_masks = []
all_ap_scores = [] 

# Define IoU thresholds
thresholds = [0.5, 0.75, 0.9]

# --- 2. THE MAIN LOOP ---
for raw_name in raw_files:
    file_id = get_id(raw_name)
    if file_id not in mask_map:
        continue
        
    # Load and Preprocess
    img = np.squeeze(imread(os.path.join(raw_dir, raw_name)))
    if img.ndim == 3: img = img[0] 
    
    gt_mask = np.squeeze(imread(os.path.join(mask_dir, mask_map[file_id])))
    if gt_mask.ndim == 3: gt_mask = gt_mask[0]
    gt_mask = label(gt_mask) 

    # Predict with tweaked parameters
    img_norm = normalize(img, 1, 99.8)
    labels, _ = model.predict_instances(img_norm, 
                                        prob_thresh=0.6, # Adjusted example
                                        nms_thresh=0.1,   # Adjusted example
                                        n_tiles=(2, 2))
    
    # Calculate stats for this specific image
    image_ap = []
    # We use thresh=0.5 specifically for the "Localization" counts
    stats_at_50 = matching_dataset([gt_mask], [labels], thresh=0.5, show_progress=False)
    
    for t in thresholds:
        stat = matching_dataset([gt_mask], [labels], thresh=t, show_progress=False)
        image_ap.append(stat.accuracy)
    
    mAP_val = np.mean(image_ap)
    
    # --- PER-IMAGE SUMMARY PRINT ---
    #print(f"--- ID {file_id} Summary ---")
    #print(f"  Cells in GT:      {stats_at_50.n_true}")
    #print(f"  Cells Localized:  {stats_at_50.n_pred} (Correct matches: {stats_at_50.tp})")
    #print(f"  mAP [0.5:0.9]:    {mAP_val:.3f}")
    #print(f"  AP @ 0.50:        {image_ap[0]:.3f}")
    #print(f"  AP @ 0.75:        {image_ap[1]:.3f}")
    #print(f"  AP @ 0.90:        {image_ap[2]:.3f}")
    #print("-" * 25)

    # Store for aggregate analysis
    all_ap_scores.append(image_ap)
    all_gt_masks.append(gt_mask)
    all_pred_masks.append(labels)
    
    # Save individual prediction
    if not os.path.exists(save_dir): os.makedirs(save_dir)
    imwrite(os.path.join(save_dir, f"pred_{file_id}.tif"), labels.astype(np.uint16))

# --- 3. AGGREGATE PERFORMANCE METRICS ---
ap_array = np.array(all_ap_scores)
mean_ap = ap_array.mean(axis=0)

print("\n" + "="*40)
print("       AGGREGATE PERFORMANCE METRICS")
print("="*40)
for i, t in enumerate(thresholds):
    print(f"mAP @ IoU {t}: {mean_ap[i]:.3f}")

# Counting uniquely labeled cells for summary
gt_counts = [len(np.unique(m)) - 1 for m in all_gt_masks]
pred_counts = [len(np.unique(m)) - 1 for m in all_pred_masks]

print("\nDATA ANALYSIS SUMMARY")
print(f"Total Bacteria (ground truth): {sum(gt_counts)}")
print(f"Total Bacteria (StarDist):     {sum(pred_counts)}")

# Average error per image
avg_error = np.mean([abs(g - p) for g, p in zip(gt_counts, pred_counts)])
print(f"Average Error per Image:       {avg_error:.1f} cells")

100%|██████████| 4/4 [00:00<00:00,  9.04it/s]


       AGGREGATE PERFORMANCE METRICS
mAP @ IoU 0.5: 0.314
mAP @ IoU 0.75: 0.046
mAP @ IoU 0.9: 0.000

DATA ANALYSIS SUMMARY
Total Bacteria (ground truth): 34
Total Bacteria (StarDist):     28
Average Error per Image:       1.0 cells
